# 09_Business Insights

## 1. Objectifs et périmètre de l'analyse

Ce notebook constitue la synthèse décisionnelle du projet AdventureWorks Advanced Analytics.

Il vise à rapprocher les résultats des analyses commerciales, produits et clients afin de dégager une vision globale de la performance de l'activité.

L'objectif est de transformer les résultats statistiques et descriptifs obtenus dans les notebooks précédents en constats métier, en opportunités commerciales et en recommandations opérationnelles.

L'analyse s'appuie principalement sur les travaux réalisés dans les notebooks :

- `04_Exploratory_Data_Analysis`
- `05_Sales_Performance`
- `06_Product_Analytics`
- `07_Customer_Analytics`
- `08_Advanced_Analytics`

Les recommandations formulées devront être directement justifiées par les résultats observés. Les estimations de coûts et de bénéfices seront systématiquement distinguées des données comptables réelles.

## Démarche d'analyse

La synthèse suivra quatre étapes :

1. Rassembler les principaux résultats des analyses précédentes.
2. Identifier les relations et les éventuelles complémentarités entre les dimensions commerciales, produits et clients.
3. Dégager les principaux constats et les opportunités d'amélioration.
4. Formuler des recommandations accompagnées d'indicateurs permettant d'en suivre les effets.

## 2. Synthèse des performances commerciales

Cette section présente une vue d'ensemble de l'activité commerciale d'AdventureWorks sur la période 2014-2016.

Elle s'appuie sur les indicateurs suivants :

- Le chiffre d'affaires total.
- Le nombre de commandes et de clients acheteurs.
- Les quantités vendues.
- Le coût total estimé et le bénéfice brut estimé.
- Le taux de marque estimé.
- Le chiffre d'affaires moyen par commande.

L'objectif est d'évaluer le niveau global de l'activité et d'identifier les principales évolutions annuelles.

Les indicateurs de rentabilité restent des estimations calculées à partir des coûts standards.

In [11]:

# Importer les bibliothèques nécessaires
from pathlib import Path
import pandas as pd
import numpy as np

# Définir le chemin du projet
BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# Charger la table des ventes
fact_sales = pd.read_csv(
    PROCESSED_DIR / "FactInternetSales.csv",
    parse_dates=["OrderDate"]
)

print("Chargement des ventes terminé.")
print("Nombre de lignes :", len(fact_sales))


Chargement des ventes terminé.
Nombre de lignes : 58189


In [15]:

# Calculer les indicateurs commerciaux globaux
synthese_commerciale = pd.DataFrame({
    "Indicateur": [
        "Nombre de lignes de vente",
        "Nombre de commandes",
        "Nombre de clients acheteurs",
        "Nombre de produits vendus",
        "Quantité totale vendue",
        "Chiffre d'affaires",
        "Coût total estimé",
        "Bénéfice brut estimé",
        "Taux de marque estimé",
        "Chiffre d'affaires moyen par commande"
    ],
    "Valeur": [
        len(fact_sales),
        fact_sales["SalesOrderNumber"].nunique(),
        fact_sales["CustomerKey"].nunique(),
        fact_sales["ProductKey"].nunique(),
        fact_sales["OrderQuantity"].sum(),
        fact_sales["SalesAmount"].sum(),
        fact_sales["CalculatedTotalCost"].sum(),
        (
            fact_sales["SalesAmount"].sum()
            - fact_sales["CalculatedTotalCost"].sum()
        ),
        (
            (
                fact_sales["SalesAmount"].sum()
                - fact_sales["CalculatedTotalCost"].sum()
            )
            / fact_sales["SalesAmount"].sum()
        ) * 100,
        (
            fact_sales["SalesAmount"].sum()
            / fact_sales["SalesOrderNumber"].nunique()
        )
    ]
})

display(synthese_commerciale.round(2))


,Indicateur,Valeur
0,Nombre de lignes de vente,58189.00
1,Nombre de commandes,26683.00
2,Nombre de clients acheteurs,17918.00
3,Nombre de produits vendus,158.00
4,Quantité totale vendue,91321.00
5,Chiffre d'affaires,29307836.59
6,Coût total estimé,27073480.62
7,Bénéfice brut estimé,2234355.98
8,Taux de marque estimé,7.62
9,Chiffre d'affaires moyen par commande,1098.37


### 2.1 Évolution annuelle des performances commerciales

Cette analyse compare les principaux indicateurs commerciaux entre 2014, 2015 et 2016.

Elle permet d'identifier les évolutions du chiffre d'affaires, du volume des commandes, des quantités vendues et de la rentabilité estimée.

Une attention particulière sera portée à l'année 2016, dont la structure des commandes présente des différences importantes par rapport aux deux années précédentes.

In [21]:

# Ajouter l'année à chaque vente
fact_sales["Annee"] = fact_sales["OrderDate"].dt.year

# Regrouper les ventes par année
synthese_annuelle = (
    fact_sales
    .groupby("Annee")
    .agg(
        Nombre_Lignes=("SalesOrderLineNumber", "count"),
        Nombre_Commandes=("SalesOrderNumber", "nunique"),
        Nombre_Clients=("CustomerKey", "nunique"),
        Quantite_Vendue=("OrderQuantity", "sum"),
        Chiffre_Affaires=("SalesAmount", "sum"),
        Cout_Total_Estime=("CalculatedTotalCost", "sum")
    )
    .reset_index()
)

# Calculer les indicateurs complémentaires
synthese_annuelle["Benefice_Brut_Estime"] = (
    synthese_annuelle["Chiffre_Affaires"]
    - synthese_annuelle["Cout_Total_Estime"]
)

synthese_annuelle["Taux_Marque_Estime_%"] = (
    synthese_annuelle["Benefice_Brut_Estime"]
    / synthese_annuelle["Chiffre_Affaires"]
) * 100

synthese_annuelle["CA_Moyen_Commande"] = (
    synthese_annuelle["Chiffre_Affaires"]
    / synthese_annuelle["Nombre_Commandes"]
)

# Calculer l'évolution annuelle du chiffre d'affaires
synthese_annuelle["Evolution_CA_%"] = (
    synthese_annuelle["Chiffre_Affaires"]
    .pct_change() * 100
)

display(synthese_annuelle.round(2))


,Annee,Nombre_Lignes,Nombre_Commandes,Nombre_Clients,Quantite_Vendue,Chiffre_Affaires,Cout_Total_Estime,Benefice_Brut_Estime,Taux_Marque_Estime_%,CA_Moyen_Commande,Evolution_CA_%
0,2014,2206,2206,2206,3492,7072084.24,6635822.93,436261.31,6.17,3205.84,NaN
1,2015,3222,3222,3222,4945,5762134.30,5200982.35,561151.94,9.74,1788.37,-18.52
2,2016,52761,21255,17412,82884,16473618.05,15236675.33,1236942.72,7.51,775.05,185.89


### 2.2 Synthèse des performances commerciales

L'analyse des performances commerciales sur la période 2014-2016 met en évidence une évolution contrastée de l'activité.

Le chiffre d'affaires global atteint 29,31 millions, pour 26 683 commandes et 17 918 clients acheteurs. Le bénéfice brut estimé s'élève à 2,23 millions, avec un taux de marque estimé de 7,62 %.

En 2015, le chiffre d'affaires recule de 18,52 % par rapport à 2014. Malgré cette baisse, le taux de marque estimé progresse de 6,17 % à 9,74 %, ce qui traduit une amélioration de la rentabilité relative estimée.

L'année 2016 marque une forte accélération de l'activité. Le chiffre d'affaires progresse de 185,89 % par rapport à 2015 et atteint 16,47 millions. Le nombre de commandes passe à 21 255, tandis que le bénéfice brut estimé atteint environ 1,24 million.

Cette croissance s'accompagne cependant d'une baisse du chiffre d'affaires moyen par commande, qui passe de 1 788,37 en 2015 à 775,05 en 2016. Cette évolution est à rapprocher de l'augmentation du nombre moyen de lignes par commande, qui atteint environ 2,48 en 2016, contre une seule ligne en moyenne en 2014 et 2015.

Enfin, le taux de marque estimé de 2016 s'établit à 7,51 %, en dessous du niveau observé en 2015.

Dans l'ensemble, les résultats montrent une forte expansion commerciale en 2016, accompagnée d'une modification de la structure des commandes et d'une rentabilité relative estimée inférieure à celle de 2015.

Ces observations devront être rapprochées des analyses produits et clients afin de mieux comprendre les facteurs associés à ces évolutions.

## 3. Synthèse de l'analyse produits

L'analyse des produits met en évidence une forte concentration de l'activité commerciale sur la catégorie des vélos, qui représente environ 96,6 % du chiffre d'affaires total.

Cette prédominance confirme le rôle central des vélos dans le modèle commercial d'AdventureWorks. Elle révèle également une dépendance importante du chiffre d'affaires à cette catégorie.

Les taux de marque estimés présentent toutefois des différences significatives entre les catégories. Les accessoires affichent le taux de marque estimé le plus élevé, à 41,06 %, contre 6,86 % pour les vélos et 5,27 % pour les vêtements.

Ces résultats montrent que la catégorie qui génère le plus de chiffre d'affaires n'est pas nécessairement celle qui présente la meilleure rentabilité relative estimée.

L'analyse détaillée des produits a également identifié une contribution négative des maillots et des casquettes, dont le déficit brut estimé cumulé atteint environ 40 680,67.

Cette situation mérite une attention particulière. Elle peut justifier un examen plus approfondi des prix de vente, des coûts standards et des volumes commercialisés pour ces produits. Les données disponibles ne permettent cependant pas, à elles seules, d'identifier les causes de cette performance.

Dans l'ensemble, l'analyse produits fait ressortir trois constats :

- Les vélos constituent le principal moteur du chiffre d'affaires.
- Les accessoires présentent le meilleur taux de marque estimé et pourraient représenter un axe de développement commercial.
- Certains produits vestimentaires, notamment les maillots et les casquettes, présentent une rentabilité estimée négative.

Ces observations soulignent l'intérêt de ne pas piloter la performance du catalogue uniquement à partir du chiffre d'affaires. La contribution au bénéfice brut estimé et le taux de marque doivent également être pris en compte.

**Limite méthodologique :** les coûts et bénéfices sont estimés à partir des coûts standards disponibles dans les données. Ils ne représentent pas nécessairement les coûts réels ni la rentabilité comptable des produits.

## 4. Synthèse de l'analyse clients

L'analyse de la clientèle met en évidence une population de 18 484 clients dans la dimension `DimCustomer`, parmi lesquels 17 918 ont effectué au moins un achat enregistré dans `FactInternetSales` sur la période étudiée.

Les clients âgés de 45 à 64 ans représentent environ 59,55 % de la population analysée. Cette tranche d'âge constitue donc une composante importante de la clientèle d'AdventureWorks.

L'analyse des performances par âge montre des différences entre les groupes. Les clients de 45 à 54 ans génèrent le chiffre d'affaires total et le chiffre d'affaires par client les plus élevés. Les clients de 55 à 64 ans présentent, quant à eux, le bénéfice brut estimé par client le plus important.

Le revenu annuel apparaît également associé à des différences de comportement d'achat. Les groupes à revenus élevés et très élevés affichent des paniers moyens supérieurs, le groupe à très hauts revenus présentant un panier moyen de 1 373,07.

D'autres caractéristiques sociodémographiques font apparaître des différences de dépenses. Les clients célibataires présentent une dépense moyenne supérieure à celle des clients mariés. Les niveaux d'éducation Bachelors et Partial College ainsi que les professions Professional et Skilled Manual figurent parmi les catégories les plus représentées.

Ces résultats permettent de mieux caractériser la clientèle et de distinguer les groupes selon leur contribution commerciale. Ils restent toutefois descriptifs et ne démontrent pas de relation causale entre les caractéristiques sociodémographiques et les dépenses.

Dans l'ensemble, l'analyse fait ressortir trois constats :

- Les clients âgés de 45 à 64 ans constituent une part importante de la clientèle.
- Les groupes d'âge et de revenus ne présentent pas tous les mêmes niveaux de dépenses et de bénéfice brut estimé par client.
- Les caractéristiques sociodémographiques peuvent aider à construire des profils commerciaux, à condition de ne pas les interpréter comme des facteurs déterminants du comportement individuel.

**Limites méthodologiques :** les indicateurs de dépenses couvrent les achats enregistrés dans `FactInternetSales` entre 2014 et 2016. Les 566 clients absents de cette table de faits peuvent avoir effectué des achats par d'autres canaux. Les bénéfices restent estimés à partir des coûts standards et les différences observées entre profils ne permettent pas d'établir des relations de causalité.

## 5. Analyse transversale

Cette section croise les résultats des analyses commerciales, produits et clients afin d'identifier les principales relations entre la performance des ventes, la rentabilité estimée des produits et la valeur commerciale des clients.

L'analyse porte notamment sur :

- La concentration du chiffre d'affaires autour des catégories de produits.
- Les différences de valeur commerciale entre les profils de clients.
- L'évolution du volume des commandes par rapport à leur valeur moyenne.
- Les points de vigilance liés à la rentabilité estimée et à la structure des ventes.

L'objectif est de dégager des constats utiles à la prise de décision, sans attribuer de relations de causalité lorsque les données disponibles ne permettent pas de les démontrer.

### 5.1 Performance commerciale et catégories de produits

L'analyse commerciale montre une forte concentration du chiffre d'affaires autour des vélos, qui représentent environ **96,6 % du chiffre d'affaires total**.

Cette concentration indique que la performance commerciale globale dépend fortement de cette catégorie. Une évolution défavorable des ventes de vélos pourrait donc avoir un effet important sur le chiffre d'affaires global.

L'analyse de rentabilité estimée révèle toutefois des différences importantes entre les catégories :

- Les accessoires présentent le meilleur taux de marque estimé, avec **41,06 %**.
- Les vélos affichent un taux de marque estimé de **6,86 %**.
- Les vêtements présentent un taux de marque estimé de **5,27 %**.

Ces résultats mettent en évidence deux dimensions complémentaires : les vélos constituent le principal moteur du chiffre d'affaires, tandis que les accessoires présentent une meilleure rentabilité estimée en proportion de leur chiffre d'affaires.

Il serait donc pertinent d'étudier les possibilités de développement des ventes d'accessoires, tout en surveillant la rentabilité des différentes catégories.

**Point de vigilance :** ces résultats ne permettent pas, à eux seuls, de démontrer que les accessoires peuvent compenser une baisse des ventes de vélos. Leur contribution absolue au bénéfice estimé doit également être prise en compte.

### 5.2 Profils clients et valeur commerciale

L'analyse des clients met en évidence plusieurs différences de valeur commerciale selon les caractéristiques sociodémographiques.

Les clients âgés de **45 à 64 ans représentent 59,55 % des clients acheteurs**. Au sein de cette population :

- Les clients de 45 à 54 ans génèrent le chiffre d'affaires total le plus élevé et présentent le chiffre d'affaires moyen par client le plus important.
- Les clients de 55 à 64 ans affichent le bénéfice brut estimé moyen par client le plus élevé.

Le niveau de revenu constitue également un critère intéressant. Les groupes à revenus élevés et très élevés présentent des paniers moyens supérieurs aux autres groupes. Le panier moyen des clients à revenus très élevés atteint notamment **1 373,07**.

Ces observations suggèrent plusieurs pistes commerciales :

- Adapter les actions commerciales aux différents profils de clients âgés de 45 à 64 ans.
- Étudier les comportements d'achat des clients à revenus élevés et très élevés.
- Examiner les caractéristiques des clients qui génèrent le plus de chiffre d'affaires et ceux qui présentent la meilleure rentabilité estimée.

Il faut toutefois éviter de considérer les caractéristiques sociodémographiques comme des explications directes du comportement d'achat. Elles constituent des critères de segmentation à croiser avec les habitudes d'achat et les produits commandés.

**Point de vigilance :** les indicateurs de rentabilité par client reposent sur des coûts estimés à partir des coûts standards des produits.

### 5.3 Volume des commandes et valeur moyenne

L'analyse annuelle met en évidence une évolution importante de la structure commerciale entre 2014 et 2016.

En 2015, le chiffre d'affaires diminue de **18,52 %** par rapport à 2014. En 2016, il augmente de **185,89 %**, pour atteindre environ **16,47 millions**.

Cette progression s'accompagne d'une forte augmentation du nombre de commandes, qui passe de 3 222 en 2015 à 21 255 en 2016.

Cependant, le chiffre d'affaires moyen par commande évolue dans le sens inverse :

- 2014 : 3 205,84
- 2015 : 1 788,37
- 2016 : 775,05

En parallèle, le nombre moyen de lignes par commande passe d'une ligne en 2014 et 2015 à environ **2,48 lignes en 2016**.

Ces résultats montrent que la croissance du chiffre d'affaires en 2016 s'accompagne d'une augmentation beaucoup plus importante du nombre de commandes et d'une diminution du chiffre d'affaires moyen par commande.

Cette évolution mérite une attention particulière. Elle peut notamment justifier une analyse plus approfondie de la composition des paniers, des quantités achetées et des catégories de produits commandées.

**Point de vigilance :** les données disponibles permettent de constater cette évolution, mais pas d'en identifier directement les causes. Il serait notamment incorrect d'attribuer la baisse du panier moyen à un changement de gamme de produits sans analyse complémentaire.

### 5.4 Principaux points de vigilance

Les analyses croisées font ressortir plusieurs éléments qui méritent une attention particulière dans le pilotage de l'activité commerciale.

**1. Une forte dépendance aux ventes de vélos**

Les vélos représentent environ 96,6 % du chiffre d'affaires total. Cette concentration constitue un point de vigilance, car la performance globale est fortement liée à une seule catégorie de produits.

**2. Des différences importantes de rentabilité estimée**

Les taux de marque estimés varient sensiblement selon les catégories. Les accessoires présentent un taux de 41,06 %, contre 6,86 % pour les vélos et 5,27 % pour les vêtements. Certains produits, notamment les maillots et les casquettes, présentent également un bénéfice brut estimé négatif.

**3. Une diminution du chiffre d'affaires moyen par commande**

Malgré la forte croissance du chiffre d'affaires en 2016, le chiffre d'affaires moyen par commande diminue nettement. Cette évolution mérite d'être surveillée afin de mieux comprendre la composition des paniers et la valeur des transactions.

**4. Une concentration de la valeur commerciale sur certains profils de clients**

Les clients âgés de 45 à 64 ans représentent une part importante de la clientèle acheteuse. Les niveaux de revenus sont également associés à des différences de panier moyen. Ces constats peuvent guider la segmentation commerciale, sans pour autant établir de relation de causalité.

**5. Des limites dans l'interprétation de la rentabilité**

Les coûts utilisés dans les analyses sont des estimations basées sur les coûts standards des produits. Les bénéfices et taux de marque présentés ne doivent donc pas être assimilés à des résultats comptables réels.

Ces points de vigilance constituent des axes de surveillance pour les analyses et les décisions commerciales futures.

## 6. Principaux enseignements métier

Les analyses réalisées sur les ventes, les produits et les clients permettent de dégager plusieurs enseignements concernant la performance commerciale de l'entreprise.

### 6.1 Une activité commerciale fortement portée par les vélos

Les vélos représentent environ 96,6 % du chiffre d'affaires total. Cette catégorie constitue le principal moteur de l'activité commerciale.

L'entreprise doit préserver sa performance sur cette gamme tout en évaluant les possibilités de diversification de ses revenus.

### 6.2 Une rentabilité estimée très variable selon les catégories

Les accessoires présentent un taux de marque estimé nettement supérieur à celui des vélos et des vêtements. À l'inverse, certains produits, notamment les maillots et les casquettes, affichent une contribution estimée négative.

L'analyse de la rentabilité doit donc être menée à la fois au niveau des catégories et des produits individuels.

### 6.3 Une croissance commerciale accompagnée d'une baisse du panier moyen

L'année 2016 se caractérise par une forte progression du chiffre d'affaires et du nombre de commandes. Toutefois, le chiffre d'affaires moyen par commande diminue sensiblement.

Le suivi de ces deux indicateurs est nécessaire pour distinguer la croissance liée à l'augmentation du nombre de transactions de celle liée à la valeur des commandes.

### 6.4 Une clientèle présentant des profils de valeur différents

Les clients âgés de 45 à 64 ans occupent une place importante dans la clientèle acheteuse. Les groupes de revenus élevés présentent également des paniers moyens supérieurs.

Ces différences offrent des possibilités de segmentation et d'adaptation des actions commerciales.

### 6.5 Une nécessité de piloter simultanément le chiffre d'affaires et la rentabilité

Une progression du chiffre d'affaires ne garantit pas automatiquement une amélioration de la rentabilité.

Le suivi commercial doit donc intégrer plusieurs dimensions : le chiffre d'affaires, le nombre de commandes, le panier moyen, les quantités vendues et les indicateurs de rentabilité estimée.

**Synthèse :** les principaux enjeux concernent la sécurisation des ventes de vélos, le développement des catégories à meilleure rentabilité estimée, la surveillance du panier moyen et une meilleure exploitation des différences de valeur entre les profils de clients.

## 7. Opportunités commerciales

Les résultats des analyses permettent d'identifier plusieurs opportunités susceptibles d'améliorer la performance commerciale, la rentabilité et la connaissance des clients.

### 7.1 Développer les ventes d'accessoires

Les accessoires présentent un taux de marque estimé de 41,06 %, nettement supérieur à celui des vélos et des vêtements.

**Opportunité :** étudier le potentiel de développement de cette catégorie, notamment à travers des offres complémentaires lors de l'achat d'un vélo.

**Indicateurs à suivre :**
- Chiffre d'affaires des accessoires.
- Quantités vendues.
- Bénéfice brut estimé.
- Taux de marque estimé.
- Part des commandes contenant au moins un accessoire.

### 7.2 Améliorer la rentabilité des produits déficitaires

Les maillots et les casquettes présentent un bénéfice brut estimé négatif, avec un déficit cumulé d'environ 40 680,67.

**Opportunité :** examiner les prix de vente, les coûts standards, les volumes commercialisés et les remises appliquées afin d'identifier les leviers d'amélioration.

**Indicateurs à suivre :**
- Bénéfice brut estimé par produit.
- Taux de marque estimé.
- Prix de vente moyen.
- Remise moyenne.
- Quantités vendues.

### 7.3 Renforcer la connaissance des clients à forte valeur commerciale

Les clients âgés de 45 à 54 ans présentent le chiffre d'affaires total et le chiffre d'affaires moyen par client les plus élevés. Les clients de 55 à 64 ans affichent le bénéfice brut estimé moyen par client le plus important.

**Opportunité :** approfondir l'étude de ces segments afin de mieux comprendre leurs habitudes d'achat et d'adapter les actions commerciales.

**Indicateurs à suivre :**
- Chiffre d'affaires par segment.
- Chiffre d'affaires moyen par client.
- Bénéfice brut estimé par client.
- Fréquence d'achat.
- Panier moyen.

### 7.4 Étudier les leviers d'amélioration du panier moyen

En 2016, le chiffre d'affaires moyen par commande s'établit à 775,05, contre 1 788,37 en 2015.

**Opportunité :** analyser les compositions de paniers et évaluer des actions commerciales telles que les ventes complémentaires, les offres groupées ou les promotions ciblées.

**Indicateurs à suivre :**
- Panier moyen.
- Nombre moyen de lignes par commande.
- Quantité moyenne par commande.
- Chiffre d'affaires par commande.
- Taux d'adoption des offres complémentaires.

### 7.5 Diversifier les sources de chiffre d'affaires

La forte concentration du chiffre d'affaires autour des vélos représente une dépendance commerciale à surveiller.

**Opportunité :** évaluer le potentiel de développement des accessoires et des vêtements, tout en préservant la performance de la catégorie principale.

**Indicateurs à suivre :**
- Part du chiffre d'affaires par catégorie.
- Évolution du chiffre d'affaires par catégorie.
- Bénéfice brut estimé par catégorie.
- Taux de marque estimé par catégorie.

Ces opportunités constituent des pistes d'action à évaluer. Leur mise en œuvre devra tenir compte des contraintes commerciales, opérationnelles et financières de l'entreprise.

## 8. Recommandations stratégiques

À partir des résultats obtenus, plusieurs recommandations peuvent être formulées afin d'améliorer le pilotage commercial, de renforcer la rentabilité estimée et de mieux exploiter les données clients.

### 8.1 Sécuriser la performance de la catégorie des vélos

Compte tenu de sa contribution de 96,6 % au chiffre d'affaires, la catégorie des vélos doit faire l'objet d'un suivi commercial prioritaire.

**Actions recommandées :**
- Suivre régulièrement l'évolution des ventes par produit.
- Identifier les références qui contribuent le plus au chiffre d'affaires.
- Surveiller les évolutions des prix, des remises et des coûts standards.
- Évaluer les risques liés à la concentration du chiffre d'affaires.

### 8.2 Développer les ventes complémentaires d'accessoires

Le taux de marque estimé des accessoires constitue un élément favorable à explorer.

**Actions recommandées :**
- Étudier les associations entre les vélos et les accessoires achetés.
- Tester des offres groupées et des ventes complémentaires.
- Mesurer leur effet sur le panier moyen et le bénéfice brut estimé.
- Évaluer les résultats avant de généraliser ces offres.

### 8.3 Examiner les produits présentant une rentabilité estimée négative

Les résultats concernant les maillots et les casquettes justifient une analyse spécifique.

**Actions recommandées :**
- Vérifier les coûts standards utilisés dans les calculs.
- Examiner les prix de vente et les remises.
- Comparer les volumes vendus et les contributions estimées.
- Étudier les possibilités d'ajustement tarifaire ou commercial.

### 8.4 Mettre en place une segmentation client orientée valeur

Les différences observées entre les groupes d'âge et de revenus peuvent contribuer à améliorer le ciblage commercial.

**Actions recommandées :**
- Identifier les segments qui génèrent le plus de chiffre d'affaires.
- Comparer la rentabilité estimée moyenne par client.
- Étudier la fréquence et la composition des achats.
- Adapter les campagnes commerciales aux comportements réellement observés.

### 8.5 Renforcer le suivi des indicateurs commerciaux

La progression du chiffre d'affaires en 2016, accompagnée d'une diminution du panier moyen, montre l'importance d'un suivi simultané des volumes et de la valeur des transactions.

**Actions recommandées :**
- Suivre le chiffre d'affaires et le nombre de commandes.
- Contrôler l'évolution du panier moyen.
- Analyser les quantités vendues et le nombre de lignes par commande.
- Comparer les performances commerciales et les indicateurs de rentabilité estimée.

### 8.6 Mettre en place un tableau de bord décisionnel

Les résultats de ce projet peuvent être exploités dans un tableau de bord Power BI destiné au suivi régulier de la performance.

Les principaux axes de pilotage recommandés sont :

- Performance commerciale : chiffre d'affaires, commandes et panier moyen.
- Performance produits : ventes, quantités et rentabilité estimée.
- Analyse clients : segmentation, valeur commerciale et comportement d'achat.
- Analyse géographique : chiffre d'affaires et rentabilité estimée par territoire.

L'objectif est de permettre aux décideurs de suivre les évolutions, de détecter les anomalies et d'orienter leurs décisions à partir d'indicateurs cohérents.

**Priorité générale :** sécuriser le chiffre d'affaires des vélos, explorer le potentiel des accessoires, corriger les situations de rentabilité estimée négative et améliorer le suivi de la valeur des commandes.

## 9. Limites méthodologiques

Malgré les analyses réalisées, plusieurs limites doivent être prises en compte pour interpréter correctement les résultats et éviter les conclusions excessives.

### 9.1 Périmètre des données commerciales

L'analyse repose principalement sur les données de ventes en ligne contenues dans la table `FactInternetSales`.

La population des clients de `DimCustomer` compte 18 484 individus, tandis que 17 918 clients apparaissent dans les ventes en ligne étudiées.

Les 566 clients absents de la table des ventes en ligne ne peuvent pas être considérés automatiquement comme des clients sans activité commerciale. Ils peuvent notamment avoir effectué des achats par d'autres canaux.

Les conclusions concernant le comportement des clients portent donc sur le périmètre des ventes en ligne disponible.

### 9.2 Limites des estimations de rentabilité

Les calculs de coûts et de bénéfices reposent sur les coûts standards des produits.

Les indicateurs présentés, notamment le bénéfice brut estimé et le taux de marque estimé, ne correspondent pas nécessairement à la rentabilité comptable réelle de l'entreprise.

Ils ne prennent pas nécessairement en compte l'ensemble des charges commerciales, logistiques, administratives et opérationnelles.

Une validation à partir des données comptables réelles serait nécessaire pour établir une analyse financière complète.

### 9.3 Limites des analyses temporelles

Les données couvrent la période 2014–2016. Cette durée permet d'observer certaines évolutions annuelles, mais reste limitée pour identifier des tendances structurelles à long terme.

L'année 2016 présente également une structure commerciale différente des deux années précédentes, notamment en matière de volume de commandes et de nombre de lignes par commande.

Cette particularité doit être prise en compte lors des comparaisons temporelles.

### 9.4 Limites des analyses clients

Les analyses sociodémographiques reposent sur les caractéristiques disponibles dans la dimension clients, notamment l'âge, le revenu annuel, la situation matrimoniale, le niveau d'études et la profession.

Ces variables permettent de décrire et de segmenter la clientèle, mais elles ne suffisent pas à expliquer les motivations d'achat.

Les relations observées sont descriptives et ne démontrent pas de lien de causalité.

### 9.5 Limites de l'interprétation des résultats

Les résultats dépendent de la qualité, de la couverture et de la granularité des données disponibles.

Certaines relations entre les produits, les clients et les commandes nécessiteraient des analyses complémentaires pour être confirmées.

Par exemple, la forte contribution des vélos au chiffre d'affaires et le panier moyen observé en 2016 ne permettent pas, à eux seuls, d'établir un lien direct entre la catégorie de produits vendue et l'évolution du montant des commandes.

### 9.6 Pistes d'amélioration

Pour approfondir cette étude, plusieurs compléments pourraient être envisagés :

- Intégrer les données de ventes réalisées par les autres canaux de distribution.
- Utiliser les coûts réels et les charges opérationnelles pour affiner l'analyse de rentabilité.
- Étendre la période d'observation pour mieux étudier les tendances temporelles.
- Enrichir l'analyse client avec des données de fréquence d'achat et de fidélité.
- Approfondir les relations entre les produits achetés, les profils clients et les montants des commandes.

Ces limites ne remettent pas en cause les constats descriptifs établis. Elles définissent le périmètre dans lequel les résultats peuvent être interprétés et les améliorations possibles pour de futures analyses.

## 10. Conclusion générale

Ce projet a permis de réaliser une analyse approfondie des données commerciales d'AdventureWorks à partir de Python, en couvrant plusieurs dimensions complémentaires : la performance des ventes, l'analyse des produits, le comportement des clients et l'exploitation des indicateurs avancés.


L'étude a porté sur 58 189 lignes de ventes, représentant un chiffre d'affaires total de 29,31 millions, pour 26 683 commandes et 17 918 clients acheteurs.

Les principaux résultats mettent en évidence :

- Une forte concentration du chiffre d'affaires autour des vélos, qui représentent environ 96,6 % des ventes.
- Des différences importantes de rentabilité estimée entre les catégories de produits, avec un taux de marque estimé particulièrement élevé pour les accessoires.
- Une progression significative du chiffre d'affaires en 2016, accompagnée d'une augmentation du nombre de commandes et d'une baisse du chiffre d'affaires moyen par commande.
- Des différences de valeur commerciale selon les profils de clients, notamment en fonction de l'âge et du niveau de revenu.
- Des possibilités d'amélioration concernant les ventes complémentaires, les produits présentant une rentabilité estimée négative et le suivi de la valeur des commandes.

Au-delà des résultats chiffrés, ce travail a permis de transformer des données brutes en informations exploitables pour le pilotage commercial. Les constats formulés constituent une base pour orienter les décisions, identifier les points de vigilance et définir des axes d'amélioration.

Le projet a également mobilisé plusieurs compétences techniques : manipulation et préparation des données avec Pandas, analyses statistiques et commerciales, visualisation des résultats et formulation de recommandations métier.

### Perspectives

La prochaine étape consiste à exploiter les résultats de cette analyse dans Power BI afin de construire un tableau de bord interactif.

Ce tableau de bord permettra de suivre les indicateurs clés, d'explorer les performances selon différentes dimensions et de faciliter la prise de décision à travers une restitution visuelle et synthétique.

Enfin, les analyses avancées prévues dans le notebook 08 pourront compléter les résultats descriptifs, notamment par des méthodes de segmentation et des tests statistiques, lorsque ces travaux auront été effectivement réalisés.

**Ce projet constitue ainsi une démarche complète d'analyse de données, depuis la compréhension des données jusqu'à la formulation d'enseignements et de recommandations orientés métier.**